In [2]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import load_web_traffic
from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()



Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [6]:
from src.data import load_sales, load_web_traffic, load_promotions, load_order_items, load_orders
from src.features import add_time_features, add_lag_features
import pandas as pd

DATA_ROOT = project_root / "data/datathon-2026-round-1"

In [ ]:
df_promotions = load_promotions(data_root=DATA_ROOT)
df_order_items = load_order_items(data_root=DATA_ROOT)
df_order = load_orders(data_root=DATA_ROOT)

df_order_items = df_order_items.groupby("order_id").agg({"promo_id_2": "sum"}).reset_index()
df_order = df_order.merge(df_order_items, on="order_id", how="left")
df_order["promo_id_2"] = df_order["promo_id_2"].fillna(0)


ValueError: Missing column provided to 'parse_dates': 'order_date'

In [9]:
df_order.head()

,order_id,order_date,customer_id,zip_x,order_status,payment_method,device_type,order_source,zip_y,city,signup_date,gender,age_group,acquisition_channel,payment_value,installments,ship_date,delivery_date,shipping_fee,promo_id_2
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,1109,Hanoi,2020-06-06,Female,35-44,social_media,7967.54,3,2012-07-07,2012-07-11,1.37,0
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search,1330,Phu Ly,2021-11-03,Female,18-24,social_media,71163.75,1,2012-07-06,2012-07-10,2.60,0
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,1473,Lao Cai,2020-09-18,Female,35-44,direct,33660.99,3,2012-07-04,2012-07-07,2.38,0
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,2360,Son Tay,2016-05-29,Male,45-54,direct,53196.25,3,2012-07-05,2012-07-11,2.49,0
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,2886,Uong Bi,2017-07-11,Male,18-24,social_media,1597.84,1,2012-07-09,2012-07-16,25.79,0


In [ ]:
from src.pipelines import train_validate_models, fit_models_full, predict_submission_from_models
from src.models import SklearnRegressorWrapper, SklearnRegressorConfig
df_sales = load_sales(data_root=DATA_ROOT)
df_traffic = load_web_traffic(data_root=DATA_ROOT)


df_traffic_test = pd.read_csv(f"{project_root}/results/webtraffic_predictions.csv")
X_traffic_test = df_traffic_test[["date", "sessions", "unique_visitors"]].drop(columns=["date"])

df = pd.merge(df_sales, df_traffic, on="date", how="left")

df = add_time_features(df, date_col="date")

# returning_users = max(0, sessions - unique_visitors)
#df["returning_users"] = df["sessions"] - df["unique_visitors"]
#df["returning_users"] = df["returning_users"].clip(lower=0)

# returning_rate = (sessions - unique_visitors) / sessions
# df["returning_rate"] = (df["sessions"] - df["unique_visitors"]) / df["sessions"]

# df = add_lag_features(df, target_col="returning_rate", lags=[3], date_col="date")

In [96]:
df1 = df.drop(columns=['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source'])
df1 = df1[df1["date"] > "2019-01-01"]
targets = ["COGS", "Revenue"]
features = [col for col in df1.columns if col not in ["date", *targets]]
print(features)

model_config = SklearnRegressorConfig(
    model_type="xgboost", 
)
results = train_validate_models(
    df=df1,
    features=features,
    targets=targets,
    model_config=model_config,
)
results["metrics"]

['year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'is_month_end', 'is_month_start', 'is_weekend']


{'COGS': {'mae': 571515.585145548,
  'rmse': 770756.556992878,
  'mape': 24.85197494714355,
  'smape': 20.940845106955518,
  'r2': 0.7367883770799903},
 'Revenue': {'mae': 676059.9579922945,
  'rmse': 944105.6913232538,
  'mape': 24.634797557205438,
  'smape': 22.565954736836613,
  'r2': 0.7015006378625642}}

In [73]:
df_sales = load_sales(data_root=DATA_ROOT)
df_traffic = load_web_traffic(data_root=DATA_ROOT)

df_traffic_test = pd.read_csv(f"{project_root}/results/webtraffic_predictions.csv")
X_traffic_test = df_traffic_test[["date", "sessions", "unique_visitors"]].drop(columns=["date"])
df = pd.merge(df_sales, df_traffic, on="date", how="left")
df = add_time_features(df, date_col="date")
df.drop(columns=['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source'], inplace=True)
targets = ["COGS", "Revenue"]
features = [col for col in df.columns if col not in ["date", *targets]]
print(features)
model_config = SklearnRegressorConfig(
    model_type="xgboost", 
)
results = train_validate_models(
    df=df,
    features=features,
    targets=targets,
    model_config=model_config,
)
results["metrics"]

['year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'is_month_end', 'is_month_start', 'is_weekend']


{'COGS': {'mae': 505952.48581566353,
  'rmse': 694086.1668286145,
  'mape': 20.086968510331392,
  'smape': 21.07987425184929,
  'r2': 0.7707160901867441},
 'Revenue': {'mae': 585314.4008703831,
  'rmse': 812629.873432484,
  'mape': 22.379391480814636,
  'smape': 21.461053364244695,
  'r2': 0.7621165636029108}}